In [ ]:
!pip install pystac-client planetary-computer odc-stac geopandas tqdm
!pip install rioxarray
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import rioxarray
import pystac_client
import planetary_computer as pc
import odc.stac
from tqdm import tqdm
from google.colab import files


In [2]:
uploaded = files.upload()

hh = pd.read_csv("target_data_pipeline.csv")

hh_unique = hh[['hhid','gps_longitude','gps_latitude']].drop_duplicates()

hh_gdf = gpd.GeoDataFrame(
    hh_unique,
    geometry=gpd.points_from_xy(
        hh_unique.gps_longitude,
        hh_unique.gps_latitude
    ),
    crs="EPSG:4326"
)


Saving target_data_pipeline.csv to target_data_pipeline (2).csv


In [3]:
BUFFER_DEG = 0.18

hh_buffers = hh_gdf.copy()
hh_buffers["geometry"] = hh_buffers.geometry.buffer(BUFFER_DEG)


/tmp/ipython-input-9597/555771101.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hh_buffers["geometry"] = hh_buffers.geometry.buffer(BUFFER_DEG)


In [4]:
MY_BBOX = [
    hh_gdf.geometry.x.min(),
    hh_gdf.geometry.y.min(),
    hh_gdf.geometry.x.max(),
    hh_gdf.geometry.y.max(),
]


In [5]:
pc_catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)


In [6]:
search_ndvi = pc_catalog.search(
    collections=["modis-13A1-061"],
    bbox=MY_BBOX,
    datetime="2008-01-01/2016-12-31"
)

items_ndvi = list(search_ndvi.get_items())


/usr/local/lib/python3.12/dist-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


In [7]:
ds_ndvi = odc.stac.load(
    items_ndvi,
    bands=["500m_16_days_NDVI"],
    bbox=MY_BBOX,
    resolution=500
)

ndvi = ds_ndvi["500m_16_days_NDVI"] / 10000


In [8]:
ndvi_monthly = ndvi.resample(time="1M").mean()


/usr/local/lib/python3.12/dist-packages/xarray/groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


In [9]:
search_lai = pc_catalog.search(
    collections=["modis-15A2H-061"],
    bbox=MY_BBOX,
    datetime="2008-01-01/2016-12-31"
)

items_lai = list(search_lai.get_items())

ds_lai = odc.stac.load(
    items_lai,
    bands=["Lai_500m"],
    bbox=MY_BBOX,
    resolution=500
)

lai = ds_lai["Lai_500m"] * 0.1
lai_monthly = lai.resample(time="1M").mean()


/usr/local/lib/python3.12/dist-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xarray/groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


In [10]:
search_lst = pc_catalog.search(
    collections=["modis-11A2-061"],
    bbox=MY_BBOX,
    datetime="2008-01-01/2016-12-31"
)

items_lst = list(search_lst.get_items())

ds_lst = odc.stac.load(
    items_lst,
    bands=["LST_Day_1km"],
    bbox=MY_BBOX,
    resolution=1000
)

lst = (ds_lst["LST_Day_1km"] * 0.02) - 273.15
lst_monthly = lst.resample(time="1M").mean()


/usr/local/lib/python3.12/dist-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xarray/groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


In [11]:
def zonal_mean(xr_data, buffers):

    results = []

    for idx, row in tqdm(buffers.iterrows(), total=len(buffers)):

        clipped = xr_data.rio.clip(
            [row.geometry],
            crs="EPSG:4326",
            drop=True
        )

        mean_series = clipped.mean(dim=["x","y"])

        df = mean_series.to_dataframe().reset_index()
        df["hhid"] = row["hhid"]

        results.append(df)

    return pd.concat(results)


In [13]:
print(ds_ndvi.rio.crs)

PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [14]:
ndvi_monthly = ndvi_monthly.rio.write_crs(ds_ndvi.rio.crs)
lai_monthly  = lai_monthly.rio.write_crs(ds_lai.rio.crs)
lst_monthly  = lst_monthly.rio.write_crs(ds_lst.rio.crs)

modis_crs = ndvi_monthly.rio.crs
hh_modis = hh_gdf.to_crs(modis_crs)

hh_modis["geometry"] = hh_modis.geometry.buffer(20000)

In [15]:
def zonal_mean(xr_data, buffers):

    results = []

    for idx, row in tqdm(buffers.iterrows(), total=len(buffers)):

        try:
            clipped = xr_data.rio.clip(
                [row.geometry],
                crs=buffers.crs,
                drop=True
            )

            if clipped.size == 0:
                continue

            mean_series = clipped.mean(dim=["x","y"])

            df = mean_series.to_dataframe().reset_index()
            df["hhid"] = row["hhid"]

            results.append(df)

        except Exception:
            continue

    return pd.concat(results, ignore_index=True)

In [16]:
df_ndvi = zonal_mean(ndvi_monthly, hh_modis)
df_lai  = zonal_mean(lai_monthly, hh_modis)
df_lst  = zonal_mean(lst_monthly, hh_modis)

100%|██████████| 1025/1025 [00:49<00:00, 20.63it/s]


In [17]:
df_ndvi = df_ndvi.rename(columns={"500m_16_days_NDVI": "ndvi"})
df_lai  = df_lai.rename(columns={"Lai_500m": "lai"})
df_lst  = df_lst.rename(columns={"LST_Day_1km": "lst"})

In [18]:
df_modis = df_ndvi.merge(
    df_lai[["time","hhid","lai"]],
    on=["time","hhid"],
    how="left"
)

df_modis = df_modis.merge(
    df_lst[["time","hhid","lst"]],
    on=["time","hhid"],
    how="left"
)

df_modis["month"] = pd.to_datetime(df_modis["time"])
df_modis = df_modis[["hhid","month","ndvi","lai","lst"]]

In [19]:
df_modis.to_parquet(
    "modis_household_monthly_2008_2016_20km.parquet",
    index=False
)

In [20]:
df_modis.to_csv(
    "modis_household_monthly_2008_2016_20km.csv",
    index=False
)

from google.colab import files
files.download("modis_household_monthly_2008_2016_20km.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>